# no-transformer — training notebook

Trains the four learned modules (TF-IDF vectorizer, domain classifier, intent classifier, epistemic router MLP, confidence regressor) and packages them into `models.zip`.

**This notebook is meant to run on Google Colab.** Click **Runtime → Run all**. ~3–5 minutes on the free CPU runtime, no GPU required.

## 1. Setup

Clones the repo, installs missing deps, sets random seeds. If you fork the repo, edit `REPO_URL` below.

In [ ]:
REPO_URL = "https://github.com/Lachi-Amine/no-transformer-.git"

import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if Path("/content/repo").exists():
        os.system("rm -rf /content/repo")
    rc = os.system(f"git clone --depth 1 {REPO_URL} /content/repo")
    if rc != 0:
        raise SystemExit(f"git clone failed (exit {rc}). Check REPO_URL.")
    PROJECT = Path("/content/repo")
    os.system("pip install -q rank-bm25 pyyaml scikit-learn==1.8.0")
else:
    PROJECT = Path.cwd()
    while not (PROJECT / "pipeline" / "schemas.py").exists() and PROJECT.parent != PROJECT:
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
(PROJECT / "models").mkdir(exist_ok=True)

import random
import numpy as np
import torch
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

import sklearn
print(f"project: {PROJECT}")
print(f"python:  {sys.version.split()[0]}")
print(f"torch:   {torch.__version__}")
print(f"numpy:   {np.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"colab:   {IN_COLAB}")

## 2. Train domain + intent classifiers

TF-IDF + logistic regression. Saves `domain_clf.pkl`, `intent_clf.pkl`, and the shared `tfidf.pkl`.

In [ ]:
import csv
import subprocess
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Auto-derive extra labeled rows from the YAML knowledge base.
subprocess.run([sys.executable, str(PROJECT / "training/expand_dataset.py")], check=True)


def load_xy(*paths):
    X, y = [], []
    for path in paths:
        if not path.exists():
            continue
        with open(path, encoding="utf-8") as f:
            for row in csv.DictReader(f):
                X.append(row["text"])
                y.append(row["label"])
    return X, y


def train_clf(name, paths, out_path):
    X, y = load_xy(*paths)
    print(f"\n=== {name} === {len(X)} rows, {len(set(y))} classes")
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    pipe = SkPipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf",   LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    pipe.fit(Xtr, ytr)
    print(classification_report(yte, pipe.predict(Xte), zero_division=0))
    joblib.dump(pipe, out_path)
    print(f"saved {out_path}")
    return pipe


domain_paths = [
    PROJECT / "training/datasets/domains.csv",
    PROJECT / "training/datasets/domains_auto.csv",
]
intent_paths = [
    PROJECT / "training/datasets/intents.csv",
    PROJECT / "training/datasets/intents_auto.csv",
]

domain_clf = train_clf("Domain", domain_paths, PROJECT / "models/domain_clf.pkl")
intent_clf = train_clf("Intent", intent_paths, PROJECT / "models/intent_clf.pkl")

joblib.dump(domain_clf.named_steps["tfidf"], PROJECT / "models/tfidf.pkl")
print("saved tfidf.pkl")

## 3. Train epistemic router MLP

Small MLP (input → 64 → 32 → 3, softmax) trained with cross-entropy on the soft `g/y/r` labels in `epistemic.csv`.

In [ ]:
import csv
import json
import re
import numpy as np
import torch
import torch.optim as optim
from sklearn.model_selection import train_test_split

from pipeline.schemas import Classification, Query
from pipeline.features import router_features
from pipeline.router import build_mlp

TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_-]*")

def make_query(text):
    norm = text.strip().lower()
    return Query(raw=text, normalized=norm,
                 tokens=tuple(TOKEN_RE.findall(norm)), entities={})

def make_cls(domain, intent):
    return Classification(domain=domain, intent=intent,
                          domain_probs={}, intent_probs={})

rows = []
with open(PROJECT / "training/datasets/epistemic.csv", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append(r)
print(f"epistemic.csv: {len(rows)} rows")

X = np.stack([
    router_features(make_query(r["text"]), make_cls(r["domain"], r["intent"]))
    for r in rows
]).astype(np.float32)
Y = np.array(
    [[float(r["g"]), float(r["y"]), float(r["r"])] for r in rows],
    dtype=np.float32,
)
INPUT_DIM = X.shape[1]
print(f"X: {X.shape}, Y: {Y.shape}, input_dim: {INPUT_DIM}")

Xtr, Xte, ytr, yte = train_test_split(X, Y, test_size=0.2, random_state=0)
Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(ytr)
Xte_t = torch.from_numpy(Xte); yte_t = torch.from_numpy(yte)

torch.manual_seed(0)
model = build_mlp(INPUT_DIM)
opt = optim.Adam(model.parameters(), lr=1e-3)

def soft_xent(pred, target):
    return -(target * (pred + 1e-9).log()).sum(dim=-1).mean()

EPOCHS = 60
BATCH = 32
n = Xtr_t.shape[0]
best_val = float("inf")
best_state = None

for epoch in range(EPOCHS):
    model.train()
    idx = torch.randperm(n)
    epoch_loss = 0.0
    for i in range(0, n, BATCH):
        b = idx[i:i + BATCH]
        loss = soft_xent(model(Xtr_t[b]), ytr_t[b])
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * len(b)
    epoch_loss /= n
    model.eval()
    with torch.no_grad():
        val_loss = soft_xent(model(Xte_t), yte_t).item()
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}  train={epoch_loss:.4f}  val={val_loss:.4f}")

model.load_state_dict(best_state)
torch.save(model.state_dict(), PROJECT / "models/epistemic_router.pt")
(PROJECT / "models/epistemic_router.json").write_text(
    json.dumps({"input_dim": INPUT_DIM})
)
print(f"\nsaved epistemic_router.pt (best val loss {best_val:.4f})")

## 4. Train confidence regressor (real labels)

Trains a `GradientBoostingRegressor` to predict whether the system's answer to a query is correct. Two label sources are merged:

1. **`qa_pairs.csv`** — for each query, run the pipeline and label = 1 if any of the row's `expected_keywords` appear in the rendered output.
2. **`feedback.csv`** — explicit user labels from the CLI's `:good`/`:bad`/`:rate` commands. Each row already has a numeric label in `[0, 1]`; we re-run the pipeline to get fresh features and use the user's label as ground truth.

Adding feedback rows over time strictly improves the regressor on real usage patterns.

In [ ]:
import csv
import joblib
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

from pipeline.orchestrator import Pipeline as ReasonerPipeline
from pipeline.features import confidence_features
from pipeline import feedback as feedback_mod

reasoner = ReasonerPipeline()
print("loaded model status:", reasoner.status())

# 1. qa_pairs.csv: keyword-derived labels from hand-curated expected answers
qa_path = PROJECT / "training/datasets/qa_pairs.csv"
qa_rows = []
with qa_path.open(encoding="utf-8") as f:
    for r in csv.DictReader(f):
        qa_rows.append(r)
print(f"qa_pairs.csv: {len(qa_rows)} rows")

X_feat, y_target = [], []
for row in qa_rows:
    resp = reasoner.run(row["query"])
    feat = confidence_features(resp.epistemic, resp.evidence)
    rendered_low = resp.rendered.lower()
    expected = [kw.strip().lower() for kw in (row.get("expected_keywords") or "").split("|") if kw.strip()]
    label = 1.0 if any(kw in rendered_low for kw in expected) else 0.0
    X_feat.append(feat)
    y_target.append(label)

# 2. feedback.csv: explicit user labels from CLI :good / :bad / :rate
feedback_rows = feedback_mod.load(PROJECT / "training/datasets/feedback.csv")
print(f"feedback.csv: {len(feedback_rows)} rows")
for row in feedback_rows:
    resp = reasoner.run(row["query"])
    feat = confidence_features(resp.epistemic, resp.evidence)
    X_feat.append(feat)
    y_target.append(float(row["label"]))

Xc = np.stack(X_feat)
yc = np.array(y_target, dtype=np.float32)
n_pos = int((yc >= 0.5).sum())
print(f"\nconfidence training: X {Xc.shape}, y {yc.shape}")
print(f"  positive (label >= 0.5): {n_pos} / {len(yc)} ({n_pos / len(yc):.1%})")
print(f"  negative (label  < 0.5): {len(yc) - n_pos} / {len(yc)}")

reg = GradientBoostingRegressor(max_depth=3, n_estimators=200, random_state=0)
reg.fit(Xc, yc)
train_r2 = reg.score(Xc, yc)
print(f"\ntrain R^2: {train_r2:.4f}")

if len(yc) >= 25 and n_pos >= 5 and (len(yc) - n_pos) >= 5:
    cv_r2 = cross_val_score(reg, Xc, yc, cv=5, scoring="r2")
    print(f"5-fold CV R^2: mean={cv_r2.mean():.4f}  std={cv_r2.std():.4f}")

joblib.dump(reg, PROJECT / "models/confidence.pkl")
print("saved confidence.pkl")

## 5. Package and download

Writes `manifest.json`, zips `models/`, triggers a download (on Colab).

After the file downloads, on your laptop:
```bash
cd "~/Documents/GitHub/no transformer "
unzip ~/Downloads/models.zip -d models/
```

Then in your CLI, type `:reload`. `:status` should show `[ok]` for every component.

In [ ]:
import json
import time
import zipfile

models_dir = PROJECT / "models"
manifest = {
    "trained_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "files": [],
}
for f in sorted(models_dir.iterdir()):
    if f.is_file() and f.name != "manifest.json":
        manifest["files"].append({"name": f.name, "size_bytes": f.stat().st_size})

(models_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))

zip_path = PROJECT / "models.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in models_dir.iterdir():
        if f.is_file():
            z.write(f, f.name)

print(f"built {zip_path}")
print(json.dumps(manifest, indent=2))

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))